In [1]:
import os
import sys

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.getcwd()))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "DeepUnitMatch"))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
import umap.umap_ as umap
from tqdm import tqdm
import h5py

from DeepUnitMatch.utils.umap_utils import embed_data

/opt/miniconda3/envs/UMPy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Forward pass the data through the network
#
# Each entry below is one "location" (e.g. one mouse/probe/loc): `data_dir`
# is a `processed_waveforms` folder as produced by `param_fun.get_snippets(...)`
# in the main DeepUnitMatch.ipynb demo (session-index-named subfolders of
# Unit{id}_RawSpikes.npy HDF5 files). `ks_dirs` is only needed if
# `load_bombcell=True`: it's the same `KS_dirs` list used to build that
# location's `processed_waveforms` folder, index-aligned with its session
# subfolders, so Bombcell's `qMetrics/templates._bc_qMetrics.parquet` can be
# found per session.

umap_path = r"/path/to/save/UMAP/data"
load_bombcell = False

locations = [
    {
        "name": "AL031",
        "data_dir": r"/path/to/AL031/processed_waveforms",
        "ks_dirs": None,  # e.g. the KS_dirs list used to build data_dir above, if load_bombcell=True
    },
    # add one entry per mouse/probe/loc you want included in the embedding
]

embedded_1_parts, embedded_2_parts = [], []
session_id_parts, unit_id_parts, depth_parts, location_parts = [], [], [], []
bombcell_parts = [] if load_bombcell else None

for loc_idx, loc in enumerate(locations):
    result = embed_data(
        loc["data_dir"], ks_dirs=loc.get("ks_dirs"), load_bombcell=load_bombcell
    )
    embedded_1_parts.append(result["embedded_first"])
    embedded_2_parts.append(result["embedded_second"])
    session_id_parts.append(result["session_id"])
    unit_id_parts.append(result["unit_id"])
    depth_parts.append(result["depth"])
    location_parts.append(np.full(len(result["depth"]), loc_idx))
    if load_bombcell:
        bombcell_parts.append(result["bombcell"])

embedded_1 = np.concatenate(embedded_1_parts)
embedded_2 = np.concatenate(embedded_2_parts)
session_id = np.concatenate(session_id_parts)
unit_id = np.concatenate(unit_id_parts)
depths = np.concatenate(depth_parts)
location = np.concatenate(location_parts)
location_names = [loc["name"] for loc in locations]

if load_bombcell:
    bc = {
        col: sum((part[col] for part in bombcell_parts), [])
        for col in bombcell_parts[0]
    }
    pd.DataFrame(bc).to_csv(os.path.join(umap_path, "bombcell.csv"))

np.save(os.path.join(umap_path, "first_half.npy"), embedded_1)
np.save(os.path.join(umap_path, "second_half.npy"), embedded_2)
np.save(os.path.join(umap_path, "depths.npy"), depths)
np.save(os.path.join(umap_path, "location.npy"), location)

/path/to/AL031/processed_waveforms is the data directory


/Users/suyash/Projects/DeepUnitMatch/DeepUnitMatch/UnitMatchPy/DeepUnitMatch/testing/test.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(read_p

FileNotFoundError: [Errno 2] No such file or directory: '/path/to/AL031/processed_waveforms'

In [ ]:
# Compute and save UMAP embeddings

n_neighbours = 5
min_dist = 0.1
n_components = 2
metric = "euclidean"

umap_embedding = umap.UMAP(n_neighbours, n_components, metric, min_dist=min_dist, random_state=0).fit_transform(0.5 * (embedded_1 + embedded_2), axis=0)
np.save(os.path.join(umap_path, "UMAPembeddings.npy"), umap_embedding)

In [ ]:
# Plot the UMAP embeddings
single_location = None

umap_embedding = np.load(os.path.join(umap_path, "UMAPembeddings.npy"))

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
if single_location:
    labels = [
        location_names[index]
        for index in location
        if location_names[index] == single_location
    ]
    indices = [i for i in location if location_names[i] == single_location]
    umap_to_plot = np.array(
        [
            umap_embedding[i, :]
            for i in range(len(location))
            if location_names[location[i]] == single_location
        ]
    )
else:
    labels = [location_names[index] for index in location]
    indices = location
    umap_to_plot = umap_embedding

scatter = ax.scatter(
    x=umap_to_plot[:, 0], y=umap_to_plot[:, 1], s=0.3, c=indices, label=labels
)
unique_labels = dict(zip(indices, labels))  # Remove duplicates
handles = [
    plt.Line2D(
        [],
        [],
        marker="o",
        linestyle="",
        markersize=5,
        color=scatter.cmap(scatter.norm(m)),
    )
    for m in unique_labels.keys()
]

ax.legend(handles, unique_labels.values())

plt.show()

In [ ]:
# Colour code by depth

single_location = "AV008"

depths_mm = depths / 1000
single_location_idx = (
    location_names.index(single_location) if single_location in location_names else None
)
filtered_depths = [
    depths_mm[i] for i, loc_idx in enumerate(location) if loc_idx == single_location_idx
]

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
if single_location:
    labels = [
        location_names[index]
        for index in location
        if location_names[index] == single_location
    ]
    indices = [i for i in location if location_names[i] == single_location]
    umap_to_plot = np.array(
        [
            umap_embedding[i, :]
            for i in range(len(location))
            if location_names[location[i]] == single_location
        ]
    )
else:
    labels = [location_names[index] for index in location]
    indices = location
    umap_to_plot = umap_embedding

scatter = ax.scatter(
    x=umap_to_plot[:, 0],
    y=umap_to_plot[:, 1],
    s=0.3,
    c=filtered_depths,
    cmap="viridis",
    label=labels,
)
cbar = plt.colorbar(scatter, ax=ax, pad=0.01, label="Depth (mm)")

In [ ]:
# Colour code UMAP by Bombcell output
# Requires load_bombcell = True (and ks_dirs supplied per location) in the
# "Forward pass" cell above, so that `bc` was populated there.
bc_param = "waveformDuration_peakTrough"
# bc_param = "spatialDecaySlope"

umap_to_plot = umap_embedding

if bc_param == "rawAmplitude":
    df = pd.DataFrame(bc)
    df = df.loc[df["rawAmplitude"] < 500]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
elif bc_param == "nSpikes":
    df = pd.DataFrame(bc)
    df = df.loc[df["nSpikes"] < 50000]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
elif bc_param == "spatialDecaySlope":
    df = pd.DataFrame(bc)
    df = df.loc[df["spatialDecaySlope"] < 0]
    df = df.loc[df["spatialDecaySlope"] > -0.015]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
else:
    umap_to_plot = umap_embedding
    colours = bc[bc_param]

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
cmap = plt.cm.viridis
scatter = ax.scatter(
    x=umap_to_plot[:, 0],
    y=umap_to_plot[:, 1],
    s=0.3,  # Small point size
    c=colours,
    cmap=cmap,
    alpha=0.8,
)
cbar = plt.colorbar(scatter, ax=ax, pad=0.01)
cbar.set_label("Waveform Duration", fontsize=14)
ax.set_title("UMAP Visualisation Colored by Waveform Duration", fontsize=14)
ax.set_xlabel("UMAP Dimension 1", fontsize=14)
ax.set_ylabel("UMAP Dimension 2", fontsize=14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
ax = plt.gca()
ax.spines[["right", "top"]].set_visible(False)
plt.show()